# Supervised Classification Models

### This script takes the data we created in the concardance scripts, merges BRCA 1 and 2, then attempts to predict the significance of the variants(Disease Causing, Benign, or VUS). I used the Clinvar Significance labels from concordance script to test how well the model performed. 

Predicting the ClinVar significance labels allowed the use of the original REVEL scores and PolyPhen-2 prediction results without significant concern for data leakage. Because REVEL and PolyPhen-2 predictions only aligned approximately 65% of the time, their outputs were sufficiently distinct to be included as separate features in the models below.


In [1]:
#importing packages needed
import pandas as pd
import numpy as np

In [ ]:
#uploading the data
BRCAVs = pd.read_csv("C:/Users/knigh/Documents/UHD MSDA/Capstone/ML Model/2026_BRCA1and2_ConcordanceLabels_22.csv")
print(BRCAVs.shape)
BRCAVs.head()
#removing the data preview to comply with All of Us data user policies.

In [3]:
#grabbing the list of columns names for further data wrangling
BRCAVs.columns

Index(['locus', 'alleles', 'qc_AC', 'qc_AF', 'qc_AN', 'call_rate', 'n_het',
       'info_AC', 'info_AF', 'info_AN', 'vid', 'contig', 'position',
       'consequence', 'clinvar_classification', 'variant_type', 'ref_allele',
       'alt_allele', 'gvs_all_ac', 'gvs_all_an', 'gvs_all_af', 'gvs_all_sc',
       'dbsnp_rsid', 'revel', 'vid_alt', 'gvs_max_subpop', 'aa_change',
       'transcript', 'gene_symbol', 'gvs_max_ac', 'gvs_max_sc', 'gvs_max_an',
       'gnomad_max_subpop', 'protein_id', 'hgvsp', 'aa_extract', 'AA1', 'AA2',
       'Protein', 'Position', 'Ref_AA', 'Alt_AA',
       '(HumDiv) Polyphen Significance(S1)', 'Score(S1)', 'Sensitivity(S1)',
       'FPR Score(S1)', 'Specificity(S1)',
       '(HumDiv) Polyphen Significance(S2)', 'Score(S2)', 'Sensitivity(S2)',
       'FPR Score(S2)', 'Specificity(S2)', 'clinvar_CL', 'revel_CL',
       '(HumDiv) Polyphen Significance(S1)_CL',
       '(HumDiv) Polyphen Significance(S2)_CL'],
      dtype='object')

In [ ]:
#dropping un-needed columns like unique identifers, labels, redunant, and non-useful information
#following most of the same columns dropped in the anomaly models
BRCA_MD1 = BRCAVs.drop(columns=['locus', 'alleles', 'qc_AC', 'qc_AN', 'info_AC', 'info_AN', 'vid', 'contig', 'position',
       'consequence', 'variant_type', 'gvs_all_ac', 'gvs_all_an','Position',
       'dbsnp_rsid', 'vid_alt', 'aa_change','transcript', 'gene_symbol', 'gnomad_max_subpop', 'protein_id', 'hgvsp', 'aa_extract', 'AA1', 'AA2',
       '(HumDiv) Polyphen Significance(S1)', 'Score(S1)', 'Sensitivity(S1)',
       'FPR Score(S1)', 'Specificity(S1)','clinvar_CL', 'revel_CL',
       '(HumDiv) Polyphen Significance(S1)_CL',
       '(HumDiv) Polyphen Significance(S2)_CL'])

BRCA_MD1.head()
#No unique idenitifers present for participant data

,qc_AF,call_rate,n_het,info_AF,clinvar_classification,ref_allele,alt_allele,gvs_all_af,gvs_all_sc,revel,...,gvs_max_sc,gvs_max_an,Protein,Ref_AA,Alt_AA,(HumDiv) Polyphen Significance(S2),Score(S2),Sensitivity(S2),FPR Score(S2),Specificity(S2)
0,"[0.999993679047306, 6.320952693990038e-06]",0.999949,1,[1.20537694547839e-06],not provided,G,T,0.000001,1,0.562,...,1,143702,P38398,T,N,Possibly Damaging,0.882,0.0644,0.824,0.176
1,"[0.9999936792071198, 6.3207928802589e-06]",0.999975,1,[2.4107248326354284e-06],uncertain significance,C,T,0.000002,2,0.690,...,1,51006,P38398,V,M,Benign,0.063,0.16,0.937,0.063
2,"[0.999981037141918, 1.8962858081970115e-05]",0.999949,3,[3.6161482717222026e-06],"uncertain significance, likely pathogenic, pat...",C,T,0.000004,3,0.862,...,3,143700,P38398,G,D,Probably Damaging,0.997,0.0167,0.409,0.591
3,"[0.9999936792071198, 6.3207928802589e-06]",0.999975,1,[1.2053711337720885e-06],"uncertain significance, likely pathogenic, not...",C,T,0.000001,1,0.789,...,1,143704,P38398,C,Y,Possibly Damaging,0.917,0.0597,0.812,0.188
4,"[0.999981035703449, 1.89642965510266e-05]",0.999874,3,[3.6165057321615856e-06],"uncertain significance, pathogenic, not provided",A,T,0.000004,3,0.812,...,3,143688,P38398,C,S,Probably Damaging,0.986,0.0368,0.736,0.264


## Exploring the Data

In [5]:
#looking at the shape of the data
BRCA_MD1.shape

(1354, 22)

In [6]:
#looking at the meta data for the dataframe to see the data types of each variable
BRCA_MD1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1354 entries, 0 to 1353
Data columns (total 22 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   qc_AF                               1354 non-null   object 
 1   call_rate                           1354 non-null   float64
 2   n_het                               1354 non-null   int64  
 3   info_AF                             1354 non-null   object 
 4   clinvar_classification              1261 non-null   object 
 5   ref_allele                          1354 non-null   object 
 6   alt_allele                          1354 non-null   object 
 7   gvs_all_af                          1354 non-null   float64
 8   gvs_all_sc                          1354 non-null   int64  
 9   revel                               1354 non-null   float64
 10  gvs_max_subpop                      1354 non-null   object 
 11  gvs_max_ac                          1354 no

In [7]:
#looking at an overview of the numeric variables
BRCA_MD1.describe()

,call_rate,n_het,gvs_all_af,gvs_all_sc,revel,gvs_max_ac,gvs_max_sc,gvs_max_an
count,1354.000000,1354.000000,1354.000000,1354.000000,1354.000000,1354.000000,1354.000000,1354.000000
mean,0.999815,187.815362,0.002268,1396.242245,0.359276,288.415805,235.062038,142707.589365
std,0.000750,2171.141622,0.033532,17676.143486,0.207359,3896.402846,2567.682665,107683.321617
min,0.979888,0.000000,0.000001,1.000000,0.010000,1.000000,1.000000,1610.000000
25%,0.999861,1.000000,0.000002,2.000000,0.183250,1.000000,1.000000,51008.000000
50%,0.999918,2.000000,0.000006,5.000000,0.330000,2.000000,2.000000,143694.000000
75%,0.999949,5.000000,0.000025,21.000000,0.517000,9.750000,9.750000,143704.000000
max,1.000000,37066.000000,0.984719,414441.000000,0.948000,130715.000000,77105.000000,446694.000000


In [8]:
#looking at an overview of the object variables
BRCA_MD1.describe(include='object')

,qc_AF,info_AF,clinvar_classification,ref_allele,alt_allele,gvs_max_subpop,Protein,Ref_AA,Alt_AA,(HumDiv) Polyphen Significance(S2),Score(S2),Sensitivity(S2),FPR Score(S2),Specificity(S2)
count,1354,1354,1261,1354,1354,1354,1354,1354,1354,1354,1354,1354,1354,1354
unique,439,948,23,4,4,7,2,20,20,7,376,343,350,352
top,"[0.9999936789673961, 6.321032603886171e-06]",[1.2054089108648328e-06],"likely benign, uncertain significance",A,G,amr,P51587,S,R,benign,1,1,1,0
freq,60,12,643,374,393,729,906,146,124,552,144,124,124,124


In [9]:
#looking for missing values
BRCA_MD1.isna().sum()

#93 missing clinvar_classifications, Will need to drop these as clinvar is going to be our target variable

qc_AF                                  0
call_rate                              0
n_het                                  0
info_AF                                0
clinvar_classification                93
ref_allele                             0
alt_allele                             0
gvs_all_af                             0
gvs_all_sc                             0
revel                                  0
gvs_max_subpop                         0
gvs_max_ac                             0
gvs_max_sc                             0
gvs_max_an                             0
Protein                                0
Ref_AA                                 0
Alt_AA                                 0
(HumDiv) Polyphen Significance(S2)     0
Score(S2)                              0
Sensitivity(S2)                        0
FPR Score(S2)                          0
Specificity(S2)                        0
dtype: int64

In [10]:
#making sure no duplicate rows
BRCA_MD1.duplicated().sum()

#none, awesome!

np.int64(0)

## Organizing our Target Variable Clinvar_Significance

In [11]:
#looking over the value counts
BRCA_MD1["clinvar_classification"].value_counts()

#note the 'not provided' class, will be considered an NA to drop

clinvar_classification
likely benign, uncertain significance                               643
benign, likely benign, uncertain significance                       211
uncertain significance                                              207
benign, likely benign                                               100
likely benign, uncertain significance, not provided                  22
uncertain significance, not provided                                 11
uncertain significance, likely pathogenic, pathogenic                 9
benign, uncertain significance                                        9
benign                                                                8
benign, likely benign, uncertain significance, likely pathogenic      7
pathogenic                                                            6
uncertain significance, likely pathogenic                             5
likely benign                                                         5
not provided                             

In [12]:
#in the exploratory section I saw that there are 93 NA's in Clinvar_classifications
#there are also 5 'not provided' classifications shown in the value counts. We will want to drop rows with all these (98 rows)

#removing NA values
BRCA_MD1 = BRCA_MD1.dropna(subset=['clinvar_classification'])

#removing rows where the value is "not provided"
BRCA_MD1 = BRCA_MD1[BRCA_MD1['clinvar_classification'] != 'not provided']

#the dataframe should now have 98 less rows (1256)
BRCA_MD1.shape

(1256, 22)

In [13]:
#Now we need to tranform the clinvar labels into a more organized list
#Right now labels are blended together

#going to condense the classifcation labels into 3 groups - VUS, Disease Causing, and Benign
#breakdown of grouping
#VUS - anything with uncertain in the label (will run this first)
#Disease Causing - pathogenic| likely pathogenic, pathogenic| likely pathogenic, pathogenic, low penetrance
#Benign - |benign, likely benign | benign | likely benign |

#changing the labels of the clinvar_classification to match the groups made eariler
#starting with the uncertain group
BRCA_MD1.loc[BRCA_MD1["clinvar_classification"].str.contains("uncertain", na=False), "clinvar_classification"] = "VUS"

#checking the counts, should be 1131
BRCA_MD1["clinvar_classification"].value_counts()

clinvar_classification
VUS                                              1131
benign, likely benign                             100
benign                                              8
pathogenic                                          6
likely benign                                       5
likely pathogenic, pathogenic                       3
likely pathogenic, pathogenic, low penetrance       1
benign, likely pathogenic, pathogenic               1
benign, likely benign, likely pathogenic            1
Name: count, dtype: int64

In [14]:
#now changing the pathogenic rows to be disease causing
BRCA_MD1.loc[BRCA_MD1["clinvar_classification"].str.contains("pathogenic", na=False), "clinvar_classification"] = "Disease Causing"

#checking count, there should be 12
BRCA_MD1["clinvar_classification"].value_counts()

clinvar_classification
VUS                      1131
benign, likely benign     100
Disease Causing            12
benign                      8
likely benign               5
Name: count, dtype: int64

In [15]:
#Lastly changing the benign group
BRCA_MD1.loc[BRCA_MD1["clinvar_classification"].str.contains("benign",na=False),"clinvar_classification"] = 'Benign'

#checking count, should be 113
BRCA_MD1["clinvar_classification"].value_counts()


#VUS = 90%
#Benign = 9%
#Disease Causing = 1%
#very imbalanced data... gonna be fun

clinvar_classification
VUS                1131
Benign              113
Disease Causing      12
Name: count, dtype: int64

### Cleaning the data

In [16]:
#looking over the data again, 23 columns
BRCA_MD1.head()


,qc_AF,call_rate,n_het,info_AF,clinvar_classification,ref_allele,alt_allele,gvs_all_af,gvs_all_sc,revel,...,gvs_max_sc,gvs_max_an,Protein,Ref_AA,Alt_AA,(HumDiv) Polyphen Significance(S2),Score(S2),Sensitivity(S2),FPR Score(S2),Specificity(S2)
1,"[0.9999936792071198, 6.3207928802589e-06]",0.999975,1,[2.4107248326354284e-06],VUS,C,T,0.000002,2,0.690,...,1,51006,P38398,V,M,Benign,0.063,0.16,0.937,0.063
2,"[0.999981037141918, 1.8962858081970115e-05]",0.999949,3,[3.6161482717222026e-06],VUS,C,T,0.000004,3,0.862,...,3,143700,P38398,G,D,Probably Damaging,0.997,0.0167,0.409,0.591
3,"[0.9999936792071198, 6.3207928802589e-06]",0.999975,1,[1.2053711337720885e-06],VUS,C,T,0.000001,1,0.789,...,1,143704,P38398,C,Y,Possibly Damaging,0.917,0.0597,0.812,0.188
4,"[0.999981035703449, 1.89642965510266e-05]",0.999874,3,[3.6165057321615856e-06],VUS,A,T,0.000004,3,0.812,...,3,143688,P38398,C,S,Probably Damaging,0.986,0.0368,0.736,0.264
5,"[0.9999936791272139, 6.320872786114306e-06]",0.999962,1,[1.2053972868917866e-06],VUS,C,T,0.000001,1,0.331,...,1,143702,P38398,V,I,Possibly Damaging,0.875,0.0653,0.826,0.174


In [17]:
#looking at the object columns
BRCA_MD1.describe(include='object')

,qc_AF,info_AF,clinvar_classification,ref_allele,alt_allele,gvs_max_subpop,Protein,Ref_AA,Alt_AA,(HumDiv) Polyphen Significance(S2),Score(S2),Sensitivity(S2),FPR Score(S2),Specificity(S2)
count,1256,1256,1256,1256,1256,1256,1256,1256,1256,1256,1256,1256,1256,1256
unique,435,926,3,4,4,7,2,20,20,6,357,329,336,338
top,"[0.9999936789673961, 6.321032603886171e-06]",[1.2054089108648328e-06],VUS,A,G,amr,P51587,S,R,benign,1,0.00018,0.00026,0.99974
freq,56,10,1131,348,370,643,841,138,112,510,134,115,115,115


In [18]:
#starting with object variables that should be numeric

#modifying info_AF
print("info_AF count of more than 1 value in cell:\n", BRCA_MD1['info_AF'].str.count(',').value_counts())
BRCA_MD1['info_AF'] = BRCA_MD1['info_AF'].astype(str).str.strip('[]').astype(float)

BRCA_MD1[['info_AF']].head()

info_AF count of more than 1 value in cell:
 info_AF
0    1256
Name: count, dtype: int64


,info_AF
1,0.000002
2,0.000004
3,0.000001
4,0.000004
5,0.000001


In [19]:
#modifying qc_AF
#ast package can safely parse a string that represents a Python literal, like a list, into the actual Python object.
import ast

#Converting the string representation of lists into actual Python lists
BRCA_MD1['qc_AF'] = BRCA_MD1['qc_AF'].apply(ast.literal_eval)

#checking the length of the lists to ensure they are all the same for when we split them later
print(BRCA_MD1['qc_AF'].apply(len).value_counts())

#This step takes each list in qc_AF, converts them into separate columns (Ref and Alt) by turning the lists into a DataFrame, 
#and aligns them with the original DataFrame’s index.
BRCA_MD1[['qc_AF_Ref', 'qc_AF_Alt']] = pd.DataFrame(BRCA_MD1['qc_AF'].tolist(), index=BRCA_MD1.index)

#Checking the values processed correctly 
BRCA_MD1[['qc_AF_Ref', 'qc_AF_Alt']].head()

qc_AF
2    1256
Name: count, dtype: int64


,qc_AF_Ref,qc_AF_Alt
1,0.999994,0.000006
2,0.999981,0.000019
3,0.999994,0.000006
4,0.999981,0.000019
5,0.999994,0.000006


In [20]:
#dropping the original qc_AF column
BRCA_MD1 = BRCA_MD1.drop(columns=["qc_AF"])

In [21]:
#Score(S2), Sensitivity(S2), FPR Score(S2), Specificity(S2) should all be numeric

num_cols = ["Score(S2)", "Sensitivity(S2)", "FPR Score(S2)", "Specificity(S2)"]

BRCA_MD1[num_cols] = BRCA_MD1[num_cols].apply(pd.to_numeric, errors="coerce")

#Checking the values processed correctly 
BRCA_MD1[["Score(S2)", "Sensitivity(S2)", "FPR Score(S2)", "Specificity(S2)"]].head()

,Score(S2),Sensitivity(S2),FPR Score(S2),Specificity(S2)
1,0.063,0.1600,0.937,0.063
2,0.997,0.0167,0.409,0.591
3,0.917,0.0597,0.812,0.188
4,0.986,0.0368,0.736,0.264
5,0.875,0.0653,0.826,0.174


In [22]:
#noticed (HumDiv) Polyphen Significance(S2) has 6 unique values, it should only have 3
BRCA_MD1["(HumDiv) Polyphen Significance(S2)"].value_counts()


(HumDiv) Polyphen Significance(S2)
benign                510
probably damaging     231
Benign                199
possibly damaging     123
Possibly Damaging     100
Probably Damaging      93
Name: count, dtype: int64

In [23]:
#for (HumDiv) Polyphen Significance(S2) we are just going to make them all lowercase to align
BRCA_MD1["(HumDiv) Polyphen Significance(S2)"] = BRCA_MD1["(HumDiv) Polyphen Significance(S2)"].str.strip().str.lower()

#checking
BRCA_MD1["(HumDiv) Polyphen Significance(S2)"].value_counts()


(HumDiv) Polyphen Significance(S2)
benign               709
probably damaging    324
possibly damaging    223
Name: count, dtype: int64

In [24]:
#looking over the object variables again before splitting into training and testing sets
BRCA_MD1.describe(include='object')

,clinvar_classification,ref_allele,alt_allele,gvs_max_subpop,Protein,Ref_AA,Alt_AA,(HumDiv) Polyphen Significance(S2)
count,1256,1256,1256,1256,1256,1256,1256,1256
unique,3,4,4,7,2,20,20,3
top,VUS,A,G,amr,P51587,S,R,benign
freq,1131,348,370,643,841,138,112,709


## Splitting the data into training, validation, and testing

In [25]:
#want to split data into train(70%), validation(15%), and test(15%)
#importing train_test_split for easy splitting
from sklearn.model_selection import train_test_split

#separating the features and target
X = BRCA_MD1.drop(columns=["clinvar_classification"])
y = BRCA_MD1["clinvar_classification"]

#First split: train and temp(validation + test) X=predictors y=labels
#stratify ensures the 90/9/1 class proportions of my labels stay similar in all datasets.
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

#Second split: validation vs test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Train: (879, 22)
Validation: (188, 22)
Test: (189, 22)


In [26]:
#checking the label proportions
print("Train proportions")
print(y_train.value_counts(normalize=True))

print("\nValidation proportions")
print(y_val.value_counts(normalize=True))

print("\nTest proportions")
print(y_test.value_counts(normalize=True))

Train proportions
clinvar_classification
VUS                0.901024
Benign             0.089875
Disease Causing    0.009101
Name: proportion, dtype: float64

Validation proportions
clinvar_classification
VUS                0.898936
Benign             0.090426
Disease Causing    0.010638
Name: proportion, dtype: float64

Test proportions
clinvar_classification
VUS                0.899471
Benign             0.089947
Disease Causing    0.010582
Name: proportion, dtype: float64


## tranforming the data for the models

In [27]:
#I need to hot code the object variables for the model
#looking at the object variables again
X_train.describe(include='object')

,ref_allele,alt_allele,gvs_max_subpop,Protein,Ref_AA,Alt_AA,(HumDiv) Polyphen Significance(S2)
count,879,879,879,879,879,879,879
unique,4,4,7,2,19,20,3
top,A,G,amr,P51587,S,R,benign
freq,249,265,450,580,100,76,498


In [28]:
#columns needing dummies
dummy_cols = ['ref_allele','alt_allele','gvs_max_subpop','Protein','Ref_AA','Alt_AA','(HumDiv) Polyphen Significance(S2)']


#creating dummy variables for all sets
X_train = pd.get_dummies(X_train, columns=dummy_cols, dtype=int)
X_test  = pd.get_dummies(X_test,  columns=dummy_cols, dtype=int)
X_val   = pd.get_dummies(X_val,   columns=dummy_cols, dtype=int)

#aligning the columns of test and val to train to ensure all sets have the same columns for the model
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)
X_val  = X_val.reindex(columns=X_train.columns, fill_value=0)

#checking the shape of the sets again
print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)


Train: (879, 74)
Validation: (188, 74)
Test: (189, 74)


In [29]:
#now need to tranform the labels to numeric

#importing label encoder to make this process more safe and easy
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

#this will turn my orignial catagories of Benign, Disease Causing, and VUS into 0,1,2
y_train_enc = le.fit_transform(y_train)
y_val_enc   = le.transform(y_val)
y_test_enc  = le.transform(y_test)

print(le.classes_)

['Benign' 'Disease Causing' 'VUS']


## Random Forest Model

In [69]:
#importing the RandomForest package
from sklearn.ensemble import RandomForestClassifier

#setting the model parameters (tried many parameters and could not predict the rare class)
rf = RandomForestClassifier(
    n_estimators=1000,
    class_weight={0:50, 1:100, 2:1}, 
    random_state=42,
    n_jobs=-1
)

#fitting the model
rf.fit(X_train, y_train_enc)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",1000
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metri

In [ ]:
#seeing how it performed on the validation set
#importing classification report for a easier to understand overview
from sklearn.metrics import classification_report

#predicting with the validation set 
val_preds = rf.predict(X_val)

#looking at performance
print(classification_report(y_val_enc, val_preds))

#tried to use many different parameters and even did oversampling and could not capture class 1(Disease Causing). Sample size is just too small for
#random forest *removed the oversampling because it didn't help and just made the script a longer read

              precision    recall  f1-score   support

           0       0.88      0.41      0.56        17
           1       0.00      0.00      0.00         2
           2       0.93      0.99      0.96       169

    accuracy                           0.93       188
   macro avg       0.60      0.47      0.51       188
weighted avg       0.92      0.93      0.92       188



c:\Users\knigh\anaconda3\envs\Python\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\knigh\anaconda3\envs\Python\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\knigh\anaconda3\envs\Python\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.sh

## Training the XGBoost Model

In [ ]:
#importing the xgboost package
#!pip install xgboost
from xgboost import XGBClassifier

#adding class weights agagin for multi-class imbalance (making class 1 matter more)
#creating a weights dictionary to apply to the model and sample weights
weights = {0: 1, 1: 100, 2: 1}

xgb = XGBClassifier(
   n_estimators=600,      # number of trees (more trees = more learning)
    learning_rate=0.05,   # how fast the model learns (lower = more careful)
    max_depth=20,          # how deep each tree can go (controls complexity)
    subsample=0.9,        # use 90% of rows per tree (adds randomness)
    colsample_bytree=0.9, # use 90% of features per tree (more randomness)
    objective='multi:softmax',  # multi-class classification, output class labels
    num_class=3,          # we have 3 classes (0, 1, 2)
    tree_method='hist',   # fast histogram-based training
    random_state=42,      # reproducibility
    min_child_weight=1,   # stops the model from making splits on tiny, noisy groups, improves generalization (added on the oversampling first then tried here, doesnt help)
    gamma=0.1,            # requires a split to reduce loss by at least this amount, prevents unnecessary splits (added on the oversampling first then tried here, doesnt help)
    scale_pos_weight=None # not used for multi-class, so leave as None, Because scale_pos_weight only works for binary classification
)
#many different combinations of parameters were attempted, this set produced the best results


#applying weights to the labels as well(tell the model class 1 is more important)
sample_weights = [weights[y] for y in y_train_enc]

#fitting the model on the training data
xgb.fit(X_train, y_train_enc, sample_weight=sample_weights)

#evaluate
val_preds = xgb.predict(X_val)
print(classification_report(y_val_enc, val_preds))

              precision    recall  f1-score   support

           0       0.88      0.41      0.56        17
           1       0.20      0.50      0.29         2
           2       0.94      0.97      0.95       169

    accuracy                           0.91       188
   macro avg       0.67      0.63      0.60       188
weighted avg       0.92      0.91      0.91       188



In [ ]:
#oversampling to try to capture more rare classes!
from imblearn.over_sampling import RandomOverSampler

#using random oversampling to try to capture the rare class more
#this creates sample copies to balance the data where all classes are balanced
ros = RandomOverSampler(random_state=42)
X_train_res, y_train_res = ros.fit_resample(X_train, y_train_enc)

#same model set up as before
weights = {0:1, 1:50, 2:1}
sample_weights = [weights[y] for y in y_train_res]

xgb2 = XGBClassifier(
    n_estimators=600,
    learning_rate=0.05,
    max_depth=20,
    subsample=0.9,
    colsample_bytree=0.9,
    objective='multi:softmax',
    num_class=3,
    tree_method='hist',
    min_child_weight=1, # added these new parameters for this model, they do seem to improve performance a tiny bit, otherwise this model performs slightly worst than the last
    gamma=0.1,          # added these new parameters for this model, they do seem to improve performance a tiny bit, otherwise this model performs slightly worst than the last
    random_state=42
)

xgb2.fit(X_train_res, y_train_res, sample_weight=sample_weights)

val_preds = xgb2.predict(X_val)
print(classification_report(y_val_enc, val_preds))

              precision    recall  f1-score   support

           0       0.70      0.41      0.52        17
           1       0.20      0.50      0.29         2
           2       0.94      0.96      0.95       169

    accuracy                           0.90       188
   macro avg       0.61      0.62      0.58       188
weighted avg       0.91      0.90      0.90       188



## Predicting testing data with both XGboosting models

In [83]:
#predicting with the first gxboost model
test_preds = xgb.predict(X_test)
print(classification_report(y_test_enc, test_preds))

              precision    recall  f1-score   support

           0       0.50      0.24      0.32        17
           1       0.00      0.00      0.00         2
           2       0.92      0.97      0.94       170

    accuracy                           0.89       189
   macro avg       0.47      0.40      0.42       189
weighted avg       0.87      0.89      0.88       189



In [93]:
#predicting with the second gxboost model
test_preds2 = xgb2.predict(X_test)
print(classification_report(y_test_enc, test_preds2))

              precision    recall  f1-score   support

           0       0.50      0.35      0.41        17
           1       0.00      0.00      0.00         2
           2       0.93      0.96      0.94       170

    accuracy                           0.89       189
   macro avg       0.48      0.44      0.45       189
weighted avg       0.88      0.89      0.88       189



In [94]:
#predicting with the Random Forest model
test_preds3 = rf.predict(X_test)
print(classification_report(y_test_enc, test_preds3))

              precision    recall  f1-score   support

           0       0.80      0.24      0.36        17
           1       0.00      0.00      0.00         2
           2       0.92      0.99      0.95       170

    accuracy                           0.92       189
   macro avg       0.57      0.41      0.44       189
weighted avg       0.90      0.92      0.89       189



c:\Users\knigh\anaconda3\envs\Python\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\knigh\anaconda3\envs\Python\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\knigh\anaconda3\envs\Python\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.sh